<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/GeoSHAP_%EC%84%9C%EB%B2%84%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# GeoShapley Young Population - Server Version
# 전체 데이터 사용 버전
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from geoshapley import GeoShapleyExplainer

import folium
import branca.colormap as cm
import osmnx as ox
import geopandas as gpd


# ============================================================
# 0. 설정
# ============================================================

YEAR = 2023   # 분석 연도 수정 가능

# 서버에 올린 엑셀 파일 이름
# main.py와 Seoul_Aprtment_FINAL.xlsx가 같은 폴더에 있어야 함
file_path = "Seoul_Aprtment_FINAL.xlsx"

# 결과 저장 파일명
excel_output_path = f"GeoShapley_YoungPop_Map_{YEAR}.xlsx"
html_output_path = f"GeoShapley_YoungPop_Map_{YEAR}.html"


# ============================================================
# 1. 데이터 불러오기
# ============================================================

print("데이터 불러오는 중...")

df = pd.read_excel(file_path)

print("원본 데이터 shape:", df.shape)


# ============================================================
# 2. 분석 연도 필터링
# ============================================================

df = df[df["Year_Sold"] == YEAR].copy()

print(f"{YEAR}년 데이터 shape:", df.shape)


# ============================================================
# 3. 변수 설정
# ============================================================

target = "Log_Price_per_m2"

features = [
    "Population",
    "Sex_ratio",
    "Pop. Density",
    "Old Population",
    "Young Population",
    "Parking_per_Household",
    "Log_Dist_Water",
    "Log_Dist_Green",
    "Log_Dist_Subway",
    "Dist_CBD",
    "max_floor",
    "heating",
    "num_of_people",
    "Bus_Stop",
    "High_School_Count"
]

coord_cols = ["Latitude", "Longitude"]

geo_features = features + coord_cols


# ============================================================
# 4. 필요한 컬럼 존재 여부 확인
# ============================================================

required_cols = geo_features + [target, "Year_Sold"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"다음 컬럼이 데이터에 없습니다: {missing_cols}")

print("필요한 컬럼 확인 완료")


# ============================================================
# 5. 문자형 변수 처리
# ============================================================

print("문자형 변수 처리 중...")

for col in geo_features:
    if df[col].dtype == "object":
        df[col] = df[col].astype("category").cat.codes

print("문자형 변수 처리 완료")


# ============================================================
# 6. 결측치 제거
# ============================================================

df_model = df.dropna(subset=geo_features + [target]).reset_index(drop=True)

X = df_model[geo_features].copy()
y = df_model[target].copy()

print("최종 X shape:", X.shape)
print("최종 y shape:", y.shape)
print("좌표 컬럼 확인:", X.columns[-2:].tolist())

if len(X) == 0:
    raise ValueError("결측치 제거 후 남은 데이터가 없습니다. YEAR 또는 컬럼 결측치를 확인하세요.")


# ============================================================
# 7. Train/Test Split
# ============================================================

print("Train/Test Split 진행 중...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


# ============================================================
# 8. XGBoost 모델 학습
# ============================================================

print("XGBoost 모델 학습 시작...")

model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

model.fit(X_train, y_train)

print("XGBoost 모델 학습 완료")


# ============================================================
# 9. GeoShapley 실행
# 전체 데이터 설명용
# ============================================================

BACKGROUND_N = 300   # 기준 표본 수

background = X_train.sample(
    n=min(BACKGROUND_N, len(X_train)),
    random_state=42
).values

# 전체 데이터 사용 유지
X_explain = X.copy()

print("Background shape:", background.shape)
print("X_explain shape:", X_explain.shape)
print("GeoShapley 실행 시작...")
print("주의: 전체 데이터 기준이라 시간이 오래 걸릴 수 있습니다.")

explainer = GeoShapleyExplainer(
    model.predict,
    background
)

geo_result = explainer.explain(X_explain)

print("GeoShapley 실행 완료")


# ============================================================
# 10. Young Population SVC 추출
# ============================================================

young_idx = geo_features.index("Young Population")

print("Young Population index:", young_idx)

young_svc = geo_result.get_svc([young_idx]).copy()

print("SVC 결과 type:", type(young_svc))
print("SVC 결과 shape:", young_svc.shape)
print("SVC 앞 5개 값:")
print(young_svc[:5])


# ============================================================
# 11. SVC 값 추출
# ============================================================

young_value = young_svc[:, 0]

print("Young Population SVC 추출 완료")
print("SVC min:", np.min(young_value))
print("SVC max:", np.max(young_value))
print("SVC mean:", np.mean(young_value))


# ============================================================
# 12. 지도 시각화 데이터 생성
# ============================================================

lat_col = "Latitude"
lon_col = "Longitude"

map_df = pd.DataFrame({
    "Latitude": X_explain[lat_col].values,
    "Longitude": X_explain[lon_col].values,
    "YoungPop_GeoShapley_SVC": young_value
})

print("지도용 데이터 shape:", map_df.shape)


# ============================================================
# 13. 서울특별시 경계 가져오기
# ============================================================

print("서울특별시 경계 데이터 가져오는 중...")

try:
    seoul_gdf = ox.geocode_to_gdf("Seoul, South Korea")
    seoul_gdf = seoul_gdf.to_crs(epsg=4326)
    has_boundary = True
    print("서울 경계 데이터 로드 완료")

except Exception as e:
    print("서울 경계 데이터를 가져오지 못했습니다.")
    print("지도는 점 데이터만 저장합니다.")
    print("오류 내용:", e)
    has_boundary = False


# ============================================================
# 14. Folium 지도 생성
# ============================================================

print("Folium 지도 생성 중...")

seoul_map = folium.Map(
    location=[37.5665, 126.9780],
    zoom_start=11,
    tiles="cartodbpositron"
)

# 서울특별시 윤곽선 추가
if has_boundary:
    folium.GeoJson(
        seoul_gdf,
        name="Seoul Boundary",
        style_function=lambda feature: {
            "fillColor": "none",
            "color": "black",
            "weight": 2.5,
            "fillOpacity": 0
        }
    ).add_to(seoul_map)


# ============================================================
# 15. 색상 범위 설정
# ============================================================

svc = map_df["YoungPop_GeoShapley_SVC"]

abs_max = max(abs(svc.min()), abs(svc.max()))

if abs_max == 0:
    abs_max = 1e-9

vmin = -abs_max
vmax = abs_max

colormap = cm.LinearColormap(
    colors=["blue", "white", "red"],
    vmin=vmin,
    vmax=vmax,
    caption="GeoShapley SVC: Young Population"
)


# ============================================================
# 16. 지도에 점 찍기
# ============================================================

print("지도에 점 추가 중...")

for _, row in map_df.iterrows():
    value = row["YoungPop_GeoShapley_SVC"]

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=1.7,
        color=colormap(value),
        fill=True,
        fill_color=colormap(value),
        fill_opacity=0.75,
        popup=f"YoungPop SVC: {value:.4f}"
    ).add_to(seoul_map)

colormap.add_to(seoul_map)
folium.LayerControl().add_to(seoul_map)

print("지도 생성 완료")


# ============================================================
# 17. 결과 엑셀 저장
# ============================================================

print("결과 엑셀 저장 중...")

result_df = pd.DataFrame({
    "Year": YEAR,
    "Latitude": X_explain["Latitude"].values,
    "Longitude": X_explain["Longitude"].values,
    "YoungPop_GeoShapley_SVC": young_value,
    "Actual_Log_Price": y.loc[X_explain.index].values,
    "Pred_Log_Price": model.predict(X_explain)
})

result_df.to_excel(excel_output_path, index=False)

print("엑셀 저장 완료:", excel_output_path)


# ============================================================
# 18. 지도 HTML 저장
# ============================================================

print("지도 HTML 저장 중...")

seoul_map.save(html_output_path)

print("지도 저장 완료:", html_output_path)


# ============================================================
# 19. 최종 완료 메시지
# ============================================================

print("============================================")
print("전체 작업 완료")
print("엑셀 결과 파일:", excel_output_path)
print("지도 HTML 파일:", html_output_path)
print("============================================")